# human in the loop (HITL)
- HITL midedleware lets you add human oversight to tool calls
- some tool calls are risky, like deleting a db or table
- checking each tool call against a configurable policy
- interrupt the flow, saves the state in persistent layer
- decision types
    - approve: execute the tool with original arguments as preposed by the agent. eg: send an email as drafted by the agent.
    - edit: modify the tool arguments before execution. eg: change the recipient before sening an email.
    - reject: skip execution and return rejection feedabck to the agent. Deny file deletion and explain why.
    - respond: Return human message directly as a synthetic tool result, skipping exectuion, for "ask user" stype tool. eg: answer an "ask_user" prompt with a direct reply
- The available decion is on the policy configred on `interrupt_on`
- when muliple tool calls are paused at same time, each action requries a seperate decision.
- use `reject` when denying the requested action. Use `respond` only when the human is action as a tool, such as answering an `ask_user` prompt. In `respond` the message is treated as a successful tool result.
- For `reject` and `respond` the middleware creates the tool message, since the tool does not run. it marks the message with notification metatda: the `source` is `hitl_interupt_decision` and the `data` holds the `decision`

## Configuring interupts


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware  # HITL Middleware
from langgraph.checkpoint.memory import InMemorySaver  # persistent storage


agent = create_agent(
    model="gpt-6.1-sol",
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject, respond) allowed
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                "read_data": False, # Safe operation, no approval needed
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver or MongoDBSaver.
    checkpointer=InMemorySaver(),
)